# CRITIC
---
_**CR**iteria **I**mportance **T**hrough **I**ntercriteria **C**orrelation_

In [1]:
import numpy as np
import pandas as pd

## 1. Introduction
---
The **CRITIC** method derives **criteria weights from the decision matrix**, combining variation in each criterion with conflict between criteria. It was proposed by [Diakoulaki, Mavrotas and Papayannakis (1995)](https://doi.org/10.1016/0305-0548(94)00059-H).

These weights describe information in the selected data. They do not automatically express a decision maker's priorities. Criteria selection, objectives and data quality still require judgment. Changing the alternatives can change the weights.

CRITIC determines weights. A **weighted sum model**, introduced separately below, demonstrates their use in evaluating alternatives.

### 1.1. Applications
---
The _CRITIC method_ can support data-based weighting in decisions such as:

- Transport service evaluation
- Financial performance comparison
- Site assessment
- Environmental technology comparison

Consider **6 freight carriers** bidding for the same regional lane, pallet specification and shipment volume. Compare **cost** (BRL/pallet), **on-time deliveries** (%), **intact deliveries** (%), and estimated **emissions** (kg CO₂e/pallet). Cost and emissions are minimized; the service percentages are maximized.

The data and carrier names are **illustrative**. The two service indicators deliberately have matching normalized profiles, making their redundancy easy to inspect.

In [2]:
df = pd.DataFrame({
    "cost": [84, 68, 82, 96, 74, 90],
    "on time": [97, 88, 92, 99, 95, 96],
    "intact": [99.2, 97.4, 98.2, 99.6, 98.8, 99.0],
    "emissions": [16, 28, 8, 24, 12, 10]
}, index=[
    "Metro Freight",
    "Budget Line",
    "Rail Connect",
    "Regional Express",
    "Consolidated Cargo",
    "Urban Shuttle"
])

df.style.background_gradient()

,cost,on time,intact,emissions
Metro Freight,84,97,99.200000,16
Budget Line,68,88,97.400000,28
Rail Connect,82,92,98.200000,8
Regional Express,96,99,99.600000,24
Consolidated Cargo,74,95,98.800000,12
Urban Shuttle,90,96,99.000000,10


## 2. Method steps
---

### 2.1. Objectives and informative criteria
---
Define the objective of each criterion. A criterion that is **constant across alternatives** cannot discriminate between them: exclude it before normalization and correlation, then assign it weight **0**.

At least two nonconstant criteria are needed for the correlation-based calculation. Missing or nonfinite values must be resolved before applying the method.

In [3]:
OBJECTIVES = pd.Series(["min", "max", "max", "min"], index=df.columns)

if len(df) < 2 or not np.isfinite(df.to_numpy(dtype=float)).all():
    raise ValueError("Use at least two alternatives with finite performances.")

RANGES = df.max() - df.min()
ACTIVE = RANGES[RANGES > 0].index

if len(ACTIVE) < 2:
    raise ValueError("CRITIC needs at least two nonconstant criteria.")

pd.DataFrame({
    "objective": OBJECTIVES,
    "range": RANGES,
    "active": RANGES > 0
})

,objective,range,active
cost,min,28.0,True
on time,max,11.0,True
intact,max,2.2,True
emissions,min,20.0,True


### 2.2. Normalized decision matrix
---
Map each active criterion to $[0,1]$, with larger values always preferred:

$$ \large
z_{ij}=\begin{cases}
\dfrac{x_{ij}-\min_i x_{ij}}{\max_i x_{ij}-\min_i x_{ij}}, & \text{if criterion }j\text{ is maximized}, \\
\dfrac{\max_i x_{ij}-x_{ij}}{\max_i x_{ij}-\min_i x_{ij}}, & \text{if criterion }j\text{ is minimized}.
\end{cases}
$$

Orienting costs before computing correlations ensures that conflict refers to the stated objectives.

In [4]:
df_n = (df[ACTIVE] - df[ACTIVE].min())/RANGES[ACTIVE]

for col in ACTIVE:
    if OBJECTIVES[col] == "min":
        df_n[col] = 1 - df_n[col]

df_n.style.background_gradient(vmin=0, vmax=1)

,cost,on time,intact,emissions
Metro Freight,0.428571,0.818182,0.818182,0.600000
Budget Line,1.000000,0.000000,0.000000,0.000000
Rail Connect,0.500000,0.363636,0.363636,1.000000
Regional Express,0.000000,1.000000,1.000000,0.200000
Consolidated Cargo,0.785714,0.636364,0.636364,0.800000
Urban Shuttle,0.214286,0.727273,0.727273,0.900000


### 2.3. Contrast intensity
---
Calculate the standard deviation of each normalized criterion:

$$ \large
\sigma_j=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(z_{ij}-\overline{z}_j)^2}
$$

Here, $n$ is the number of alternatives. Use the **population standard deviation**, `ddof=0`. Greater dispersion gives greater contrast intensity.

In [5]:
SIGMA = df_n.std(ddof=0)

SIGMA.to_frame("standard deviation")

,standard deviation
cost,0.333546
on time,0.326725
intact,0.326725
emissions,0.367045


### 2.4. Intercriteria correlation
---
Compute the **Pearson correlation** between every pair of normalized criteria:

$$ \large
r_{jk}=\frac{\sum_i(z_{ij}-\overline{z}_j)(z_{ik}-\overline{z}_k)}
{\sqrt{\sum_i(z_{ij}-\overline{z}_j)^2\sum_i(z_{ik}-\overline{z}_k)^2}}
$$

Positive correlation indicates aligned performances; negative correlation indicates conflict. The two service indicators have $r=1$, while lower cost conflicts with better service in this sample.

In [6]:
CORRELATION = df_n.corr(method="pearson").clip(-1, 1)

if not np.isfinite(CORRELATION.to_numpy()).all():
    raise ValueError("Correlation is undefined; review the active criteria.")

CORRELATION.style.background_gradient(vmin=-1, vmax=1, cmap="coolwarm")

,cost,on time,intact,emissions
cost,1.000000,-0.829233,-0.829233,-0.176653
on time,-0.829233,1.000000,1.000000,0.208467
intact,-0.829233,1.000000,1.000000,0.208467
emissions,-0.176653,0.208467,0.208467,1.000000


### 2.5. Conflict and information content
---
For each pair, conflict is **$1-r_{jk}$**, rather than $1-|r_{jk}|$. Perfect positive correlation contributes **0**; perfect negative correlation contributes **2**.

Combine total conflict with contrast intensity:

$$ \large
H_j=\sum_{k\in J}(1-r_{jk}), \qquad C_j=\sigma_j H_j
$$

Here $J$ contains the active criteria. Self-correlation contributes zero.

In [7]:
df_conflict = 1 - CORRELATION

df_conflict

,cost,on time,intact,emissions
cost,0.000000,1.829233e+00,1.829233e+00,1.176653
on time,1.829233,0.000000e+00,2.220446e-16,0.791533
intact,1.829233,2.220446e-16,0.000000e+00,0.791533
emissions,1.176653,7.915327e-01,7.915327e-01,0.000000


In [8]:
CONFLICT = df_conflict.sum(axis=1)
INFORMATION = SIGMA*CONFLICT

if INFORMATION.sum() <= 1e-12:
    raise ValueError("No usable CRITIC information; criteria may be fully concordant.")

pd.DataFrame({
    "standard deviation": SIGMA,
    "total conflict": CONFLICT,
    "information": INFORMATION
})

,standard deviation,total conflict,information
cost,0.333546,4.835118,1.612734
on time,0.326725,2.620765,0.856270
intact,0.326725,2.620765,0.856270
emissions,0.367045,2.759718,1.012942


### 2.6. Criteria weights
---
Normalize the information contents:

$$ \large
w_j=\frac{C_j}{\sum_{k\in J}C_k}
$$

For comparison, display **dispersion-only weights**, $\sigma_j/\sum_k\sigma_k$. The CRITIC weights also account for the correlation structure.

In [9]:
WEIGHTS = pd.Series(0.0, index=df.columns)
WEIGHTS.loc[ACTIVE] = INFORMATION/INFORMATION.sum()

pd.DataFrame({
    "dispersion-only weight": SIGMA/SIGMA.sum(),
    "CRITIC weight": WEIGHTS.loc[ACTIVE]
}).style.background_gradient()

,dispersion-only weight,CRITIC weight
cost,0.246334,0.371751
on time,0.241296,0.197378
intact,0.241296,0.197378
emissions,0.271074,0.233493


The mutually redundant service indicators receive the same weights. Their conflict with each other is zero; cost contributes a different, conflicting perspective and receives the largest weight in this matrix.

Both service indicators retain positive weights. CRITIC accounts for correlations but is **not invariant to duplicating criteria**; it does not automatically remove double counting or replace careful criterion design.

### 2.7. Application
---
Use the derived weights in a **weighted sum model** with the normalized performances:

$$ \large
S_i=\sum_{j\in J}w_jz_{ij}
$$

This is an additional aggregation choice, with linear performance scales and compensation between criteria. The following table shows each criterion's contribution to the score.

In [10]:
df_contributions = df_n*WEIGHTS.loc[ACTIVE]

df_contributions["score"] = df_contributions.sum(axis=1)

df_contributions

,cost,on time,intact,emissions,score
Metro Freight,0.159322,0.161491,0.161491,0.140096,0.622400
Budget Line,0.371751,0.000000,0.000000,0.000000,0.371751
Rail Connect,0.185875,0.071774,0.071774,0.233493,0.562916
Regional Express,0.000000,0.197378,0.197378,0.046699,0.441455
Consolidated Cargo,0.292090,0.125604,0.125604,0.186794,0.730093
Urban Shuttle,0.079661,0.143548,0.143548,0.210143,0.576900


### 2.8. Score and Ranking
---
Rank alternatives by **decreasing score**. Equal scores would share a rank.

In [11]:
df_score = df.join(df_contributions["score"])

df_score["ranking"] = (
    df_score["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

(
    df_score.sort_values("ranking")
    .style.background_gradient(subset=list(df.columns))
)

,cost,on time,intact,emissions,score,ranking
Consolidated Cargo,74,95,98.800000,12,0.730093,1
Metro Freight,84,97,99.200000,16,0.622400,2
Urban Shuttle,90,96,99.000000,10,0.576900,3
Rail Connect,82,92,98.200000,8,0.562916,4
Regional Express,96,99,99.600000,24,0.441455,5
Budget Line,68,88,97.400000,28,0.371751,6


**Consolidated Cargo** leads through a combination of low cost, good service and relatively low emissions. Budget Line is cheapest but has the weakest service and highest emissions; Regional Express leads on service but is expensive. The result follows from the displayed CRITIC weights and the stated aggregation rule.

## 3. Examples
---

### 3.1. Community garden site selection
---
Compare **4 sites** available under a municipal community-garden lease scheme. Evaluate the **annual lease fee** (BRL, minimize), **nearby households** (maximize), **usable area** (m², maximize), and **traffic noise** (dBA, minimize). All sites have an existing **water connection**, coded as 1.

The data and site names are **illustrative**. The constant connection indicator demonstrates why a nondiscriminating criterion must be excluded before correlation and assigned zero weight.

In [12]:
df_ex1 = pd.DataFrame({
    "annual fee": [2400, 3600, 1800, 3000],
    "nearby households": [1200, 1800, 700, 1500],
    "usable area": [600, 500, 800, 650],
    "traffic noise": [58, 62, 54, 52],
    "water connection": [1, 1, 1, 1]
}, index=[
    "Station Garden",
    "Market Garden",
    "Hill Garden",
    "Riverside Garden"
])

df_ex1.style.background_gradient()

,annual fee,nearby households,usable area,traffic noise,water connection
Station Garden,2400,1200,600,58,1
Market Garden,3600,1800,500,62,1
Hill Garden,1800,700,800,54,1
Riverside Garden,3000,1500,650,52,1


In [13]:
OBJECTIVES_ex1 = pd.Series(
    ["min", "max", "max", "min", "max"], index=df_ex1.columns
)

if len(df_ex1) < 2 or not np.isfinite(df_ex1.to_numpy(dtype=float)).all():
    raise ValueError("Use at least two alternatives with finite performances.")

RANGES_ex1 = df_ex1.max() - df_ex1.min()
ACTIVE_ex1 = RANGES_ex1[RANGES_ex1 > 0].index

if len(ACTIVE_ex1) < 2:
    raise ValueError("CRITIC needs at least two nonconstant criteria.")

pd.DataFrame({
    "objective": OBJECTIVES_ex1,
    "range": RANGES_ex1,
    "active": RANGES_ex1 > 0
})

,objective,range,active
annual fee,min,1800,True
nearby households,max,1100,True
usable area,max,300,True
traffic noise,min,10,True
water connection,max,0,False


In [14]:
# Normalize only the informative criteria.
df_ex1_n = (
    (df_ex1[ACTIVE_ex1] - df_ex1[ACTIVE_ex1].min())/RANGES_ex1[ACTIVE_ex1]
)

for col in ACTIVE_ex1:
    if OBJECTIVES_ex1[col] == "min":
        df_ex1_n[col] = 1 - df_ex1_n[col]

df_ex1_n.style.background_gradient(vmin=0, vmax=1)

,annual fee,nearby households,usable area,traffic noise
Station Garden,0.666667,0.454545,0.333333,0.400000
Market Garden,0.000000,1.000000,0.000000,0.000000
Hill Garden,1.000000,0.000000,1.000000,0.800000
Riverside Garden,0.333333,0.727273,0.500000,1.000000


In [15]:
# Contrast intensity and correlation
SIGMA_ex1 = df_ex1_n.std(ddof=0)
CORRELATION_ex1 = df_ex1_n.corr(method="pearson").clip(-1, 1)

if not np.isfinite(CORRELATION_ex1.to_numpy()).all():
    raise ValueError("Correlation is undefined; review the active criteria.")

CORRELATION_ex1.style.background_gradient(vmin=-1, vmax=1, cmap="coolwarm")

,annual fee,nearby households,usable area,traffic noise
annual fee,1.000000,-0.990867,0.877876,0.524000
nearby households,-0.990867,1.000000,-0.909656,-0.512805
usable area,0.877876,-0.909656,1.000000,0.766679
traffic noise,0.524000,-0.512805,0.766679,1.000000


In [16]:
# Conflict, information and weights
CONFLICT_ex1 = (1 - CORRELATION_ex1).sum(axis=1)
INFORMATION_ex1 = SIGMA_ex1*CONFLICT_ex1

if INFORMATION_ex1.sum() <= 1e-12:
    raise ValueError("No usable CRITIC information; criteria may be fully concordant.")

WEIGHTS_ex1 = pd.Series(0.0, index=df_ex1.columns)
WEIGHTS_ex1.loc[ACTIVE_ex1] = INFORMATION_ex1/INFORMATION_ex1.sum()

pd.DataFrame({
    "standard deviation": SIGMA_ex1,
    "total conflict": CONFLICT_ex1,
    "information": INFORMATION_ex1,
    "weight": WEIGHTS_ex1.loc[ACTIVE_ex1]
})

,standard deviation,total conflict,information,weight
annual fee,0.372678,2.588991,0.964860,0.208185
nearby households,0.369274,5.413328,1.999004,0.431319
usable area,0.360844,2.265101,0.817348,0.176356
traffic noise,0.384057,2.222126,0.853423,0.184140


In [17]:
WEIGHTS_ex1.to_frame("weight").style.background_gradient()

,weight
annual fee,0.208185
nearby households,0.431319
usable area,0.176356
traffic noise,0.184140
water connection,0.000000


In [18]:
# Weighted-sum contributions, separate from CRITIC weighting
df_ex1_contributions = df_ex1_n*WEIGHTS_ex1.loc[ACTIVE_ex1]
df_ex1_contributions["score"] = df_ex1_contributions.sum(axis=1)

df_ex1_contributions

,annual fee,nearby households,usable area,traffic noise,score
Station Garden,0.138790,0.196054,0.058785,0.073656,0.467285
Market Garden,0.000000,0.431319,0.000000,0.000000,0.431319
Hill Garden,0.208185,0.000000,0.176356,0.147312,0.531853
Riverside Garden,0.069395,0.313686,0.088178,0.184140,0.655400


In [19]:
df_ex1_score = df_ex1.join(df_ex1_contributions["score"])

df_ex1_score["ranking"] = (
    df_ex1_score["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

(
    df_ex1_score.sort_values("ranking")
    .style.background_gradient(subset=list(df_ex1.columns))
)

,annual fee,nearby households,usable area,traffic noise,water connection,score,ranking
Riverside Garden,3000,1500,650,52,1,0.655400,1
Hill Garden,1800,700,800,54,1,0.531853,2
Station Garden,2400,1200,600,58,1,0.467285,3
Market Garden,3600,1800,500,62,1,0.431319,4


**Riverside Garden** leads: it combines substantial household reach with the lowest noise level. Hill Garden is cheaper and larger, but reaches fewer households. The household criterion has the greatest information-based weight because it strongly conflicts with the fee and area criteria.

Water connection has weight **0**, contributes nothing to the score, and never enters the correlation matrix. If every criterion were constant, or if all active normalized profiles were perfectly aligned, the recipe would stop instead of inventing weights.

## 4. References
---

- Diakoulaki, D., Mavrotas, G. and Papayannakis, L. (1995). [*Determining objective weights in multiple criteria problems: The CRITIC method*](https://doi.org/10.1016/0305-0548(94)00059-H). *Computers & Operations Research*, 22(7), 763–770.